<a href="https://colab.research.google.com/github/supastishn/llm-test/blob/main/Qwen_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git lfs install
!git clone https://huggingface.co/Qwen/Qwen3-0.6B qwen-0.5b-base

Git LFS initialized.
Cloning into 'qwen-0.5b-base'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 71 (delta 31), reused 0 (delta 0), pack-reused 4 (from 1)
Unpacking objects: 100% (71/71), 1.74 MiB | 5.19 MiB/s, done.


In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 # Or cu121 depending on your CUDA
!pip install transformers datasets accelerate peft bitsandbytes sentencepiece Jinja2 trl --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 59.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.6 MB/s e

In [ ]:
import os
import torch
# No torch_xla imports needed
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,      # For QLoRA
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model # Added prepare_model_for_kbit_training
from trl import SFTTrainer

# --- Configuration ---
base_model_path = "./qwen-0.5b-base"  # Path to your local Qwen-0.5B files
new_model_name = "qwen-0.5b-dolly-qlora-gpu" # Name for the fine-tuned adapter
dataset_name = "databricks/databricks-dolly-15k" # Example instruction dataset
output_dir = f"./results-{new_model_name}" # Directory to save LoRA adapter & logs
use_bf16_compute = True # Use bfloat16 for computation (requires Ampere GPU like T4 or newer)

# --- Quantization Configuration (for QLoRA) ---
# Use 4-bit quantization to save memory
compute_dtype = torch.bfloat16 if use_bf16_compute else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # Use NF4 quantization type
    bnb_4bit_compute_dtype=compute_dtype, # Compute in bf16/fp16 for speed/accuracy balance
    bnb_4bit_use_double_quant=False,   # Disable double quantization
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {base_model_path}")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True
)
# Set padding token if necessary
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set pad_token to eos_token: {tokenizer.pad_token}")
tokenizer.padding_side = "right"

# --- Load Base Model (Quantized) ---
print(f"Loading base model with 4-bit quantization: {base_model_path}")
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config, # Apply quantization config
    torch_dtype=compute_dtype,      # Load weights in compute dtype (though stored in 4-bit)
    device_map="auto",              # Automatically distribute across available GPUs (uses the T4)
    trust_remote_code=True,
)
model.config.use_cache = False # Disable cache during training
model.config.pretraining_tp = 1 # Assuming no tensor parallelism needed for 0.5B model on one GPU

# --- Prepare Model for K-bit Training ---
# This prepares the quantized model for PEFT training
model = prepare_model_for_kbit_training(model)
print(f"Model loaded on {model.device}") # Should show layers mapped to cuda:0

# --- LoRA Configuration ---
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] # Verify for your Qwen version

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,                  # Start low for T4 VRAM
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules
)

# --- Apply LoRA / PEFT to the Model ---
model = get_peft_model(model, peft_config)
print("\nLoRA Model Structure:")
model.print_trainable_parameters() # Show number of trainable parameters (should be small)

# --- Load and Prepare Dataset ---
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name, split="train")

# Optional: Shuffle and select a subset for faster testing/iteration
# dataset = dataset.shuffle(seed=42).select(range(1000))

# --- Define Formatting Function (using Qwen Chat Template) ---
# (Same formatting function as before)
def format_dolly(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]
    if context:
        prompt = f"Instruction:\n{instruction}\n\nContext:\n{context}"
    else:
        prompt = f"Instruction:\n{instruction}"
    try:
        chat = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
        ]
        formatted_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
        if not formatted_text.endswith(tokenizer.eos_token):
             formatted_text += tokenizer.eos_token
        return {"text": formatted_text}
    except Exception as e:
        print(f"Chat template failed: {e}. Falling back to manual formatting.")
        system_prompt = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
        user_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n"
        assistant_prompt = f"<|im_start|>assistant\n{response}{tokenizer.eos_token}<|im_end|>"
        return {"text": system_prompt + user_prompt + assistant_prompt}

print("Formatting dataset...")
formatted_dataset = dataset.map(format_dolly, remove_columns=list(dataset.features))

print("\nSample Formatted Text:")
print(formatted_dataset[0]['text'])

# --- Training Arguments ---
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2, # Keep low for T4 VRAM
    gradient_accumulation_steps=8, # Effective batch size = 2 * 8 = 16
    optim="paged_adamw_8bit",   # Optimizer for QLoRA
    save_steps=100,
    logging_steps=1,
    learning_rate=0.001,
    weight_decay=0.001,
    fp16=False,                 # Disable fp16 when using bf16 compute dtype with bitsandbytes
    bf16=use_bf16_compute,      # Enable bf16 compute type (requires Ampere GPU like T4)
    max_grad_norm=0.3,
    max_steps=50,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    # No TPU arguments needed
)

# --- Initialize Trainer (using TRL's SFTTrainer) ---
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    #dataset_text_field="text",
  #  max_seq_length=1024,        # Adjust based on VRAM
   # tokenizer=tokenizer,
    args=training_arguments,
    #packing=False,
)

# --- Start Training ---
print("\nStarting QLoRA training on GPU...")
trainer.train()

# --- Save the LoRA Adapter ---
print(f"Saving LoRA adapter to {output_dir}")
trainer.model.save_pretrained(output_dir) # Saves only the adapter weights
tokenizer.save_pretrained(output_dir) # Save tokenizer alongside adapter

# No xm.rendezvous needed
print("\nTraining finished!")

# --- Optional: Inference Example (GPU) ---
# print("\nLoading model for inference...")
# import gc
# # Clear memory
# del model
# del trainer
# gc.collect()
# torch.cuda.empty_cache()
# gc.collect()

# # Load the base model again (quantized)
# base_model_for_inference = AutoModelForCausalLM.from_pretrained(
#     base_model_path,
#     quantization_config=bnb_config, # Need to load base quantized
#     torch_dtype=compute_dtype,
#     device_map="auto",
#     trust_remote_code=True,
# )
# # Load the PEFT model (adapter) on top of the base model
# inference_model = PeftModel.from_pretrained(base_model_for_inference, output_dir)
# inference_model.eval() # Set to evaluation mode

# # Optional: Merge adapter for faster inference (consumes more VRAM initially to merge)
# # print("Merging adapter...")
# # inference_model = inference_model.merge_and_unload()
# # print("Adapter merged.")

# # Reload tokenizer
# tokenizer_inf = AutoTokenizer.from_pretrained(output_dir, trust_remote_code=True)
# if tokenizer_inf.pad_token is None:
#     tokenizer_inf.pad_token = tokenizer_inf.eos_token

# print("Running inference example...")
# # Use pipeline or manual generation
# pipe = pipeline(task="text-generation", model=inference_model, tokenizer=tokenizer_inf, device=0, max_length=200) # Specify GPU device 0

# chat_prompt = [
#     {"role": "system", "content": "You are a helpful assistant."},
#     {"role": "user", "content": "What is the capital of France?"}
# ]
# prompt_text = tokenizer_inf.apply_chat_template(chat_prompt, tokenize=False, add_generation_prompt=True)

# result = pipe(prompt_text)
# print("\nGenerated Text:")
# print(result[0]['generated_text'])

Loading tokenizer: ./qwen-0.5b-base
Loading base model with 4-bit quantization: ./qwen-0.5b-base
Model loaded on cuda:0

LoRA Model Structure:
trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395
Loading dataset: databricks/databricks-dolly-15k
Formatting dataset...

Sample Formatted Text:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Instruction:
When did Virgin Australia start operating?

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>
<|im_start|>assistant
<

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



Starting QLoRA training on GPU...


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.184200


In [13]:
from google.colab import files
files.download('results-qwen-0.5b-dolly-qlora-gpu/')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
#import torch
'''from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
    logging,
)'''
from peft import PeftModel
import gc # For garbage collection

# --- Configuration ---
base_model_path = "./qwen-0.5b-base"  # Path to the ORIGINAL Qwen-0.5B files
adapter_path = "./results-qwen-0.5b-dolly-qlora-gpu" # IMPORTANT: Path to your saved LoRA adapter directory (where adapter_model.safetensors is)
use_bf16_compute = True # Should match the compute dtype used during training

# --- Quantization Configuration (MUST match training) ---
# Use the same bnb_config as used for fine-tuning
compute_dtype = torch.bfloat16 if use_bf16_compute else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if not torch.cuda.is_available():
    print("Warning: CUDA not found. Running on CPU will be very slow and might fail if model doesn't fit RAM.")

# --- 1. Load Tokenizer ---
# Load the tokenizer from the adapter directory to ensure consistency
print(f"Loading tokenizer from adapter directory: {adapter_path}")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
# Ensure pad token is set (it should have been saved, but double-check)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set pad_token to eos_token: {tokenizer.pad_token}")
tokenizer.padding_side = "right" # Usually needed for generation

# --- 2. Load Base Model (Quantized) ---
print(f"Loading base model with 4-bit quantization: {base_model_path}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype, # Match compute dtype
    device_map="auto",         # Automatically map to GPU
    trust_remote_code=True,
)
print(f"Base model loaded on device(s): {base_model.device}")

# --- 3. Load LoRA Adapter ---
print(f"Loading LoRA adapter: {adapter_path}")
# This command loads the adapter weights and applies them to the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
print("LoRA adapter loaded.")

# --- 4. Merge Adapter (Optional but Recommended for Inference Speed) ---
# Merging combines the LoRA weights directly into the base model's weights.
# This increases memory usage slightly *during* the merge process,
# but results in faster inference afterwards as LoRA calculations are no longer separate.
# If you are extremely memory constrained, you *could* skip this, but inference will be slower.
print("Merging LoRA adapter into base model...")
model = model.merge_and_unload()
print("Adapter merged.")

# --- 5. Set Model to Evaluation Mode ---
# Crucial for disabling dropout and ensuring consistent outputs
model.eval()
print("Model set to evaluation mode.")

# --- 6. Inference ---

# --- Method A: Using Hugging Face Pipeline (Easier) ---
print("\n--- Running Inference with Pipeline ---")
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1, # Specify GPU 0 or CPU
    # Common generation parameters:
    # max_length=512,          # Max total length (prompt + generation)
    max_new_tokens=100,       # Max tokens to generate *after* the prompt
    temperature=0.7,         # Controls randomness (lower = more deterministic)
    top_p=0.9,               # Nucleus sampling probability
    do_sample=True,          # Whether to use sampling; set to False for greedy decoding
    pad_token_id=tokenizer.eos_token_id, # Important for stopping generation
)

# --- IMPORTANT: Format your prompt using the Chat Template ---
# Use the same format the model was fine-tuned on (Qwen chat format)
chat_prompt = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short story about a robot who learns to paint."}
    # Add more turns if needed, e.g.:
    # {"role": "assistant", "content": "Okay, here's a draft..."},
    # {"role": "user", "content": "Can you make the ending happier?"}
]

# `apply_chat_template` formats the chat correctly.
# `add_generation_prompt=True` ensures it ends correctly to signal the model to start generating.
prompt_text = tokenizer.apply_chat_template(chat_prompt, tokenize=False, add_generation_prompt=True)

print(f"\nFormatted Prompt:\n{prompt_text}")

print("\nGenerating response (Pipeline)...")
result = pipe(prompt_text)
print("\nGenerated Text (Pipeline):")
# The output often includes the prompt, depending on pipeline settings.
# You might need to parse just the generated part if needed.
print(result[0]['generated_text'])


# --- Method B: Using model.generate() (More Control) ---
print("\n--- Running Inference with model.generate() ---")

# Format the prompt again (same as above)
prompt_text_generate = tokenizer.apply_chat_template(chat_prompt, tokenize=False, add_generation_prompt=True)
print(f"\nFormatted Prompt:\n{prompt_text_generate}")

# Tokenize the input
inputs = tokenizer(prompt_text_generate, return_tensors="pt").to(device)

print("\nGenerating response (model.generate)...")
# Generate text
with torch.no_grad(): # Disable gradient calculations for inference
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,      # Max tokens to generate *after* the prompt
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id # Crucial for stopping correctly
        # Add other parameters like top_k, repetition_penalty etc. if needed
    )

# Decode the generated tokens
# outputs[0] contains the token IDs for the *entire* sequence (prompt + generation)
# We slice it to get only the generated part by starting after the input length
generated_ids = outputs[0][inputs.input_ids.shape[1]:]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\nGenerated Text (model.generate):")
print(generated_text)


# --- Cleanup (Optional) ---
#If running multiple inferences or needing memory back
print("\nCleaning up...")
del model
del base_model
del pipe
gc.collect()
torch.cuda.empty_cache()
gc.collect()
print("Cleanup complete.")

Using device: cuda
Loading tokenizer from adapter directory: ./results-qwen-0.5b-dolly-qlora-gpu
Loading base model with 4-bit quantization: ./qwen-0.5b-base
Base model loaded on device(s): cuda:0
Loading LoRA adapter: ./results-qwen-0.5b-dolly-qlora-gpu
LoRA adapter loaded.
Merging LoRA adapter into base model...


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Adapter merged.
Model set to evaluation mode.

--- Running Inference with Pipeline ---


ValueError: The model has been loaded with `accelerate` and therefore cannot be moved to a specific device. Please discard the `device` argument when creating your pipeline object.